# Huấn luyện mô hình hư hỏng (hu_hong.onnx) trên Google Colab

1. Chọn **Runtime → Change runtime type → GPU (T4)**.
2. Trên trạm: trang `/label.html` → **Tải dataset (.zip)** (chỉ gồm ảnh đã duyệt).
3. Chạy lần lượt các ô bên dưới. Cuối cùng tải `hu_hong.onnx`, chép vào `SmartPK_PC/` rồi chạy lại `tram_ai`.

Mô hình giữ ở **YOLOv8/YOLO11 detect** (đầu ra `(1, 4+số_lớp, N)`, không NMS tích hợp) vì `OnnxYolo` của trạm chỉ đọc kiểu này.

In [ ]:
!pip -q install "ultralytics>=8.3.0,<8.4" onnxruntime onnx onnxslim
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ – hãy bật GPU trong Runtime")

In [ ]:
# Tải zip dataset lên (hoặc đặt ZIP_PATH tới tệp trên Google Drive đã mount)
import os
from google.colab import files
ZIP_PATH = ""            # ví dụ "/content/drive/MyDrive/dataset_hu_hong.zip"; để trống = hiện nút chọn tệp
if not ZIP_PATH:
    up = files.upload()
    ZIP_PATH = next(iter(up))
!rm -rf /content/ds_raw && mkdir -p /content/ds_raw && unzip -q -o "{ZIP_PATH}" -d /content/ds_raw
!ls /content/ds_raw

In [ ]:
# Chia train/val 80/20 (seed cố định) và ghi data.yaml với tên lớp lấy từ classes.txt
import os, random, shutil
SRC, DST, SEED, VAL = "/content/ds_raw", "/content/ds", 42, 0.2
names = [ln.strip() for ln in open(f"{SRC}/classes.txt", encoding="utf-8") if ln.strip()]
stems = sorted(f[:-4] for f in os.listdir(f"{SRC}/images") if f.endswith(".jpg") and os.path.isfile(f"{SRC}/labels/{f[:-4]}.txt"))
assert len(stems) >= 2, "Cần ít nhất 2 ảnh đã duyệt"
random.Random(SEED).shuffle(stems)
n_val = max(1, round(len(stems) * VAL))
split = {"val": stems[:n_val], "train": stems[n_val:]}
shutil.rmtree(DST, ignore_errors=True)
for sp, items in split.items():
    for sub in ("images", "labels"):
        os.makedirs(f"{DST}/{sp}/{sub}", exist_ok=True)
    for s in items:
        shutil.copy(f"{SRC}/images/{s}.jpg", f"{DST}/{sp}/images/{s}.jpg")
        shutil.copy(f"{SRC}/labels/{s}.txt", f"{DST}/{sp}/labels/{s}.txt")
with open(f"{DST}/data.yaml", "w", encoding="utf-8") as f:
    f.write(f"path: {DST}\ntrain: train/images\nval: val/images\nnames:\n")
    for i, n in enumerate(names):
        f.write(f"  {i}: {n}\n")
neg = sum(1 for s in stems if not open(f"{SRC}/labels/{s}.txt").read().strip())
print(f"{len(stems)} ảnh: train {len(split['train'])}, val {len(split['val'])}, ảnh xe lành (nhãn rỗng) {neg}; lớp: {names}")

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
res = model.train(data="/content/ds/data.yaml", imgsz=416, epochs=100, patience=20, batch=16, seed=42, project="/content/runs", name="hu_hong", exist_ok=True)
m = model.val()
print("mAP50:", m.box.map50, "mAP50-95:", m.box.map)
for i, n in m.names.items():
    print(f"  {n}: mAP50-95 = {m.box.maps[i]:.3f}")   # lớp ít ảnh sẽ thấp – thêm dữ liệu cho lớp đó

In [ ]:
# Xuất ONNX đúng kiểu OnnxYolo đọc được: ảnh vuông tĩnh 416, KHÔNG NMS tích hợp, opset 12
import onnxruntime as ort, numpy as np, shutil
best = YOLO("/content/runs/hu_hong/weights/best.pt")
path = best.export(format="onnx", imgsz=416, opset=12, nms=False, simplify=True, dynamic=False)
sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
out = sess.run(None, {sess.get_inputs()[0].name: np.zeros((1, 3, 416, 416), np.float32)})[0]
nc = len(best.names)
print("Đầu ra:", out.shape, "| mong đợi (1,", 4 + nc, ", N)")
assert out.ndim == 3 and out.shape[0] == 1 and out.shape[1] == 4 + nc, "Sai kiểu đầu ra – trạm sẽ đọc sai"
shutil.copy(path, "/content/hu_hong.onnx")
files.download("/content/hu_hong.onnx")